# momentum-buffer-update composite — cx11: one SGD-momentum step on the param, then zero_grad with set_to_none

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `momentum-buffer-update`, `zero-grad-set-none`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "momentum-buffer-update"
DD_ATOM_IDS = ["momentum-buffer-update", "zero-grad-set-none"]
DD_SUBTOPICS = ["Optimizer: Momentum buffer", "PyTorch: zero_grad"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Every training-loop iteration has three optimizer-side moves:
1. `backward()` populates `p.grad` (the caller does this).
2. `optimizer.step()` — your inner work: `momentum-buffer-update` plus the param step.
3. `optimizer.zero_grad(set_to_none=True)` — clear grads so the NEXT backward starts fresh.

**The two atoms.**
- **momentum-buffer-update** — `v <- mu * v + g`, in place on the velocity buffer. Mid-step operation: reads `p.grad`, mutates `v`.
- **zero-grad-set-none** — `p.grad = None` AFTER the step. Crucial because PyTorch's `backward()` ACCUMULATES into `p.grad` if it's already a tensor (this is how RNNs and multi-loss training work). If you forget to clear, you double-count the previous grad.

**Anatomy.**
```python
def step(self):
    for p, v in zip(self.params, self.velocity):
        v.mul_(self.mu).add_(p.grad)         # momentum-buffer-update.
        p.data.add_(v, alpha=-self.lr)
    for p in self.params:
        p.grad = None                         # zero-grad-set-none.
```

**Why both atoms together.** Without zero-grad, momentum compounds with stale grads and training diverges. Without momentum-buffer-update, you have plain SGD. The pair is the minimum viable 'one optimizer iteration' on a momentum optimizer.

### Composite Exercise — one SGD-momentum step on the param, then zero_grad with set_to_none

**Atoms exercised together**: `momentum-buffer-update`, `zero-grad-set-none`

Implement `cx11_sgd_momentum_full(params, velocity_buffers, lr, mu)`.

Assumes the caller has ALREADY called `backward()` so each `p.grad` is a tensor. The function does TWO things for each param:

1. Read `p.grad`, update the velocity buffer in place: `v <- mu * v + g`.
2. Update the param in place: `p.data.add_(v, alpha=-lr)`.
3. AFTER all params have stepped, run a SECOND pass that sets `p.grad = None` for every param (set-to-none semantics). Two passes are cleaner than one because no one should set `p.grad = None` while still reading `p.grad` mid-loop, even though here that's safe.

Return `None`. The test cross-checks the param update against `torch.optim.SGD(momentum=...)`, then verifies every `p.grad is None`, then runs a fresh backward and verifies the new grad is freshly allocated (not contaminated by a stale tensor).

**Gotcha:** if you do `v.mul_(mu).add_(p.grad)` and `p.grad = None` in the SAME loop body, you must order them correctly (read grad BEFORE setting None). The two-pass design avoids this entirely.

In [ ]:
def cx11_sgd_momentum_full(params, velocity_buffers, lr, mu):
    # Pass 1: step (atom A: momentum-buffer-update + inplace param update).
    for p, v in zip(params, velocity_buffers):
        v.mul_(mu).add_(p.grad)
        p.data.add_(v, alpha=-lr)
    # Pass 2: clear grads with set-to-none (atom B).
    for p in params:
        p.grad = None
    return None


<details><summary>Show solution — cx11</summary>

```python
def cx11_sgd_momentum_full(params, velocity_buffers, lr, mu):
    # Pass 1: step (atom A: momentum-buffer-update + inplace param update).
    for p, v in zip(params, velocity_buffers):
        v.mul_(mu).add_(p.grad)
        p.data.add_(v, alpha=-lr)
    # Pass 2: clear grads with set-to-none (atom B).
    for p in params:
        p.grad = None
    return None
```

The two-pass design mirrors `torch.optim.Optimizer.step()` + `zero_grad()` — they're separate methods for exactly this reason. Folding the grad-clear INTO the step is fine for vanilla SGD-momentum but breaks the moment you want to inspect or log `p.grad` between step and zero_grad. The set-to-none clear is also what lets your training loop do `optimizer.step(); optimizer.zero_grad(set_to_none=True)` and pay no per-param `zero_()` kernel cost.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx11'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx11',
        'subtopics': ["Optimizer: Momentum buffer", "PyTorch: zero_grad"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()